#### Glorot and He Initialization


By default, Keras uses Glorot initialization with a uniform distribution. 

When you create a layer, you can switch to He initialization by setting kernel_initializer="he_uniform" or kernel_initializer="he_normal" like this:

In [3]:
import tensorflow as tf
dense = tf.keras.layers.Dense(50, activation="relu", kernel_initializer="he_normal")


c:\Users\USER\envs\env\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


VarianceScaling

In [4]:
he_avg_init = tf.keras.initializers.VarianceScaling(scale=2.,
                                                    mode="fan_avg",
                                                    distribution="uniform")
dense = tf.keras.layers.Dense(50,
                              activation="sigmoid",
                              kernel_initializer=he_avg_init)

#### Better Activation Functions


+ Leaky ReLU

In [5]:
leaky_relu = tf.keras.layers.LeakyReLU(alpha=0.2) # defaults to alpha=0.3

dense = tf.keras.layers.Dense(50,
                              activation=leaky_relu,
                              kernel_initializer="he_normal")  #  He normalization




# If you prefer, you can also use LeakyReLU as a separate layer in your model
model = tf.keras.models.Sequential([
#  [...] # more layers
 tf.keras.layers.Dense(50, kernel_initializer="he_normal"), # no activation
 tf.keras.layers.LeakyReLU(alpha=0.2), # activation as a separate layer
#  [...] # more layers
])


c:\Users\USER\envs\env\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Batch Normalization

In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),  # Birinci təbəqə kimi BN
    tf.keras.layers.Dense(300, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),  # Gizli təbəqədən sonra BN
    tf.keras.layers.Dense(100, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),  # Gizli təbəqədən sonra BN
    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()


c:\Users\USER\envs\env\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

In [7]:
# Layer (type)                 Output Shape      Param #
# =================================================================
# flatten (Flatten)            (None, 784)       0
# _________________________________________________________________
# batch_normalization          (None, 784)       3136      ← 4 × 784 = 3136
# _________________________________________________________________
# dense (Dense)                (None, 300)       235500
# _________________________________________________________________
# batch_normalization_1        (None, 300)       1200      ← 4 × 300 = 1200
# _________________________________________________________________
# dense_1 (Dense)              (None, 100)       30100
# _________________________________________________________________
# batch_normalization_2        (None, 100)       400       ← 4 × 100 = 400
# _________________________________________________________________
# dense_2 (Dense)              (None, 10)        1010
# =================================================================
# Total params: 271,346
# Trainable params: 268,978    ← γ və β (öyrənilən)
# Non-trainable params: 2,368   ← μ və σ (hərəkətli ortalama)

In [8]:
[(var.name, var.trainable) for var in model.layers[1].variables]


[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),  # bias-ı söndür!
    tf.keras.layers.BatchNormalization(),  # BN əvvəl
    tf.keras.layers.Activation("relu"),    # Aktivləşdirmə sonra
    tf.keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [10]:
# Hər pixeli müstəqil normallaşdırmaq üçün:
tf.keras.layers.BatchNormalization(axis=[1, 2])

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'

#### Gradient Clipping

clipvalue

In [ ]:
optimizer = tf.keras.optimizers.SGD(clipvalue=1.0)

clipnorm 

In [ ]:
optimizer = tf.keras.optimizers.SGD(clipnorm=1.0)

In [ ]:
[...] # Assuming model A was already trained and saved to "my_model_A"
model_A = tf.keras.models.load_model("my_model_A")
model_B_on_A = tf.keras.Sequential(model_A.layers[:-1])
model_B_on_A.add(tf.keras.layers.Dense(1, activation="sigmoid"))


NameError: name 'tf' is not defined

In [ ]:
model_A_clone = tf.keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())


NameError: name 'tf' is not defined

In [ ]:
for layer in model_B_on_A.layers[:-1]:
 layer.trainable = False
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy", 
                     optimizer=optimizer,
                    metrics=["accuracy"])



history = model_B_on_A.fit(X_train_B,
                            y_train_B,
                            epochs=4,
                            validation_data=(X_valid_B, y_valid_B))
for layer in model_B_on_A.layers[:-1]:
 layer.trainable = True
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy",
                     optimizer=optimizer,
                        metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B, 
                           y_train_B,
                             epochs=16,
                        validation_data=(X_valid_B, y_valid_B))

model_B_on_A.evaluate(X_test_B, y_test_B)


In [12]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

In [11]:
optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.001, 
    momentum=0.9, 
    nesterov=True
)

In [ ]:
# Power Scheduling

optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.01, 
    decay=1e-4          # 1/s (s = addım sayı)
)


# Exponential Scheduling


# Metod 1: Funksiya ilə

# Sadə versiya
def exponential_decay_fn(epoch):
    return 0.01 * 0.1 ** (epoch / 20)
# Konfiqurasiya edilə bilən versiya
def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1 ** (epoch / s)
    return exponential_decay_fn
exponential_decay_fn = exponential_decay(lr0=0.01, s=20)
# Callback ilə istifadə
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn)
history = model.fit(..., callbacks=[lr_scheduler])


# Metod 2: Əvvəlki öyrənmə dərəcəsini istifadə etmək

def exponential_decay_fn(epoch, lr):
    return lr * 0.1 ** (1 / 20)  # hər epoxda 0.1^(1/20) qədər vur



# Piecewise Constant Scheduling

def piecewise_constant_fn(epoch):
    if epoch < 5:
        return 0.01
    elif epoch < 15:
        return 0.005
    else:
        return 0.001

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(piecewise_constant_fn)

#  Performance Scheduling

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    factor=0.5,      # 0.5-ə vur
    patience=5,      # 5 epoxdır yaxşılaşma yoxdursa
    verbose=1        # məlumat göstər
)

history = model.fit(..., callbacks=[lr_scheduler])

# Schedules API (tf.keras.optimizers.schedules)

import math

batch_size = 32
n_epochs = 25
n_steps = n_epochs * math.ceil(len(X_train) / batch_size)

scheduled_learning_rate = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=n_steps,    # neçə addımda
    decay_rate=0.1          # 0.1-ə vur
)

optimizer = tf.keras.optimizers.SGD(learning_rate=scheduled_learning_rate)




# 1cycle Scheduling

class OneCycleScheduler(tf.keras.callbacks.Callback):
    def __init__(self, iterations, max_lr=0.01, ...):
        # başlanğıc dəyərləri
    def on_batch_begin(self, batch, logs=None):
        # hər batch-də öyrənmə dərəcəsini yenilə
        new_lr = ...  # 1cycle formuluna görə hesabla
        tf.keras.backend.set_value(
            self.model.optimizer.learning_rate, 
            new_lr
        )

ℓ1
 and ℓ2
 Regularization

In [ ]:
layer = tf.keras.layers.Dense(100, activation="relu",
 kernel_initializer="he_normal",
 kernel_regularizer=tf.keras.regularizers.l2(0.01)) # regularization factor of 0.01 
 

layer = tf.keras.layers.Dense(100, activation="relu",
 kernel_initializer="he_normal",
 kernel_regularizer=tf.keras.regularizers.l1(0.01)) # regularization factor of 0.01


layer = tf.keras.layers.Dense(100, activation="relu",
 kernel_initializer="he_normal",
 kernel_regularizer=tf.keras.regularizers.l1_l2(0.01)) # regularization factor of 0.01

In [14]:
from functools import partial
RegularizedDense = partial(tf.keras.layers.Dense,
 activation="relu",
 kernel_initializer="he_normal",
 kernel_regularizer=tf.keras.regularizers.l2(0.01))
model = tf.keras.Sequential([
 tf.keras.layers.Flatten(input_shape=[28, 28]),
 RegularizedDense(100),
 RegularizedDense(100),
 RegularizedDense(10, activation="softmax")
])


Dropout

In [ ]:
model = tf.keras.Sequential([
 tf.keras.layers.Flatten(input_shape=[28, 28]),
 tf.keras.layers.Dropout(rate=0.2),
 tf.keras.layers.Dense(100, activation="relu",
 kernel_initializer="he_normal"),
 tf.keras.layers.Dropout(rate=0.2),
 tf.keras.layers.Dense(100, activation="relu",
 kernel_initializer="he_normal"),
 tf.keras.layers.Dropout(rate=0.2),
 tf.keras.layers.Dense(10, activation="softmax")
])
[...] # compile and train the model

Monte Carlo (MC) Dropout


In [ ]:
import numpy as np

y_probas = np.stack([model(X_test, training=True) for sample in range(100)])
y_proba = y_probas.mean(axis=0)

model.predict(X_test[:1]).round(3)
y_proba[0].round(3)
y_std = y_probas.std(axis=0)
y_std[0].round(3)


y_pred = y_proba.argmax(axis=1)
accuracy = (y_pred == y_test).sum() / len(y_test)
accuracy

In [ ]:
class MCDropout(tf.keras.layers.Dropout):
    def call(self, inputs, training=False):
        return super().call(inputs, training=True)


Max-Norm Regularization


In [ ]:
dense = tf.keras.layers.Dense(
    100, 
    activation="relu", 
    kernel_initializer="he_normal",
    kernel_constraint=tf.keras.constraints.max_norm(1.)  # r = 1
)